In [2]:
import kagglehub
import pandas as pd
import numpy as np

/home/lucasp/Estudos/Machine-learning-exercises/ml_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 01. Leitura do Dataset

In [3]:
# Download latest version
path = kagglehub.dataset_download("ranafayezz/titanic-cleaned")

df_train = pd.read_csv(path+"/train_data.csv")
df_test = pd.read_csv(path+"/test_data.csv")

# 02. Análise Exploratória

In [4]:
df_train.head()

,Unnamed: 0,PassengerId,Survived,Sex,Age,Fare,Pclass_1,Pclass_2,Pclass_3,Family_size,Title_1,Title_2,Title_3,Title_4,Emb_1,Emb_2,Emb_3
0,0,1,0,1,0.2750,0.014151,0,0,1,0.1,1,0,0,0,0,0,1
1,1,2,1,0,0.4750,0.139136,1,0,0,0.1,1,0,0,0,1,0,0
2,2,3,1,0,0.3250,0.015469,0,0,1,0.0,0,0,0,1,0,0,1
3,3,4,1,0,0.4375,0.103644,1,0,0,0.1,1,0,0,0,0,0,1
4,4,5,0,1,0.4375,0.015713,0,0,1,0.0,1,0,0,0,0,0,1


In [5]:
df_train.describe()

,Unnamed: 0,PassengerId,Survived,Sex,Age,Fare,Pclass_1,Pclass_2,Pclass_3,Family_size,Title_1,Title_2,Title_3,Title_4,Emb_1,Emb_2,Emb_3
count,792.000000,792.000000,792.000000,792.000000,792.000000,792.000000,792.000000,792.000000,792.000000,792.000000,792.000000,792.000000,792.000000,792.000000,792.000000,792.000000,792.000000
mean,395.500000,396.500000,0.386364,0.647727,0.368244,0.064677,0.243687,0.208333,0.547980,0.088636,0.744949,0.005051,0.040404,0.209596,0.185606,0.092172,0.720960
std,228.774999,228.774999,0.487223,0.477980,0.162994,0.100987,0.429577,0.406373,0.498007,0.154485,0.436165,0.070932,0.197029,0.407277,0.389034,0.289451,0.448811
min,0.000000,1.000000,0.000000,0.000000,0.008375,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,197.750000,198.750000,0.000000,0.000000,0.275000,0.015469,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,395.500000,396.500000,0.000000,1.000000,0.350000,0.028302,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,593.250000,594.250000,1.000000,1.000000,0.437500,0.061045,0.000000,0.000000,1.000000,0.100000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,791.000000,792.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


# 03. Construção do exercício

In [6]:
# Vou gerar uma variável qualitativa ordinal a partir das colunas
# `Pclass_1`, `Pclass_2`, e `Pclass_3` (qualitativas nominais)
df_train['Pclass'] = df_train['Pclass_1'] + 2*df_train['Pclass_2'] + 3*df_train['Pclass_3']

In [13]:
# Para fins desse exercício, vou escolher três variáveis explicativas:
var_explicativas = [
    "Age",  # Quantitativa discreta
    "Pclass",  # Qualitativa ordinal
    "Sex",  # Qualitativa nominal
]

df_X = df_train[var_explicativas]
df_y = df_train["Survived"]
df_train = df_train[var_explicativas + ["Survived"]]

def calculate_gini(y):
    """
    Calcula a impureza Gini de um vetor de labels (Target).
    Exemplo de uso: calculate_gini(df_train['Survived'])
    """
    n = len(y)
    if n == 0:
        return 0
        
    gini = 1.0
    for i in y.unique():
        prob = np.sum(y == i) / n
        gini -= prob ** 2
        
    return gini

# 04. Geração da stump

In [14]:
df_train.describe()

,Age,Pclass,Sex,Survived
count,792.000000,792.000000,792.000000,792.000000
mean,0.368244,2.304293,0.647727,0.386364
std,0.162994,0.836634,0.477980,0.487223
min,0.008375,1.000000,0.000000,0.000000
25%,0.275000,2.000000,0.000000,0.000000
50%,0.350000,3.000000,1.000000,0.000000
75%,0.437500,3.000000,1.000000,1.000000
max,1.000000,3.000000,1.000000,1.000000


In [28]:
# Primeiro, vou criar uma função para abstrair o algoritmo CART
def find_cart_threshold(df, variable, target):
    # Primeiro passo é definir qual é o melhor threshold dessa variável.
    # Para isso vou ordenar o data frame por ela (para visualização), e 
    # calcular a quantidade de sobreviventes e casualidades para cada valor da
    # variável, e em seguida aplicar o mesmo cálculo de Gini e obter o melhor threshold
    possible_variable_values = df[variable].unique()

    if len(possible_variable_values) == 1:
        return None, 0

    gini_split_values = {}

    for value in possible_variable_values:
        lt_subset = df[df[variable] <= value]
        gt_subset = df[df[variable] > value]

        if len(lt_subset) == 0 or len(gt_subset) == 0:
            continue

        gini_lt = calculate_gini(lt_subset[target])
        gini_gt = calculate_gini(gt_subset[target])
        gini_split_values[value] = (gini_lt * len(lt_subset) + gini_gt * len(gt_subset)) / len(df)

    best_threshold = min(gini_split_values, key=gini_split_values.get)
    return best_threshold, gini_split_values[best_threshold]

In [ ]:
# Primeiro vou encontrar a variável e o threshold que melhor se ajustam às informações
target = 'Survived'

# Calcular impurezas e armazenar em um dicionário
candidates = dict()
for var in df_train.columns:
    if var == target:
        continue
    
    candidates[var] = find_cart_threshold(df_train, var, target)

# Encontrar a variável e o threshold que possuem a menor impureza
impurities = [candidate[1] for candidate in candidates.values()]
variables = list(candidates.keys())

idx_min_gini = np.argmin(impurities)
variable_min_gini = variables[idx_min_gini]

threshold, impurity = candidates[variable_min_gini]
variable_min_gini, threshold, impurity

{'Age': (np.float64(0.075), np.float64(0.46197412959381046)),
 'Pclass': (np.int64(2), np.float64(0.4267739829102236)),
 'Sex': (np.int64(0), np.float64(0.33105087217518836))}

In [25]:
df_train_stump

,Age,Pclass,Sex,Survived
1,0.4750,1,0,1
2,0.3250,3,0,1
3,0.4375,1,0,1
8,0.3375,3,0,1
9,0.1750,2,0,1
...,...,...,...,...
777,0.0625,3,0,1
779,0.5375,1,0,1
780,0.1625,3,0,1
781,0.2125,1,0,1


In [ ]:
# Replicação da lógica para o nó esquerdo

# Definir a stump e refazer o processo
stump = variable_min_gini, threshold

# Definir novo grupo de dados
df_train_stump = df_train[df_train[variable_min_gini] <= threshold]

# Encontrar a variável e o threshold que melhor se ajustam às informações
target = 'Survived'

# Calcular impurezas e armazenar em um dicionário
candidates = dict()
for var in df_train_stump.columns:
    if var == target:
        continue
    
    threshold, impurity = find_cart_threshold(df_train_stump, var, target)
    if threshold is None:
        continue

    candidates[var] = threshold, impurity
    print(f"Variável: {var}, Threshold: {candidates[var][0]}, Impureza: {candidates[var][1]}")

# Encontrar a variável e o threshold que possuem a menor impureza
impurities = [candidate[1] for candidate in candidates.values()]
variables = list(candidates.keys())

idx_min_gini = np.argmin(impurities)
variable_min_gini = variables[idx_min_gini]

threshold, impurity = candidates[variable_min_gini]
variable_min_gini, threshold, impurity

Variável: Age, Threshold: 0.4, Impureza: 0.37126312566416314
Variável: Pclass, Threshold: 2, Impureza: 0.2834760497519524


('Pclass', np.int64(2), np.float64(0.2834760497519524))

# 05. Generalização

In [ ]:
# Construir classe que representa um nó na árvore

class Node:
    """Nó de uma árvore de decisão.

    `prediction` guarda a classe majoritária do nó (útil na pós-poda:
    ao colapsar uma subárvore, a folha herda essa predição).
    Folha = ausência de filhos (left/right is None).
    """

    def __init__(
        self,
        feature=None,
        threshold=None,
        left=None,
        right=None,
        prediction=None,
        n_samples=0,
    ):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.prediction = prediction
        self.n_samples = n_samples

    @property
    def is_leaf(self):
        return self.left is None and self.right is None

    def prune_to_leaf(self):
        """Transforma este nó em folha (subárvore descartada)."""
        self.feature = None
        self.threshold = None
        self.left = None
        self.right = None


In [31]:
# Função para abstrair o processo que executamos para encontrar o split
def find_best_split(df, target):
    """
    Entre todas as features, escolhe o (feature, threshold) com menor
    impureza Gini ponderada do split. Retorna (None, None, None) se não
    houver split válido.
    """
    candidates = {}
    for var in df.columns:
        if var == target:
            continue

        threshold, impurity = find_cart_threshold(df, var, target)
        if threshold is None:
            continue

        candidates[var] = (threshold, impurity)

    if not candidates:
        return None, None, None

    best_feature = min(candidates, key=lambda var: candidates[var][1])
    threshold, impurity = candidates[best_feature]
    return best_feature, threshold, impurity

In [ ]:
def majority_class(y):
    """Determina a predição de uma folha."""
    return y.value_counts().idxmax()


def build_tree(
    df,
    target="Survived",
    depth=0,
    max_depth=None,
    min_samples_split=2,
):
    """
    Constrói recursivamente uma árvore CART.

    Critérios de parada (folha):
      - nó puro (uma única classe)
      - menos de min_samples_split amostras
      - profundidade >= max_depth (se informado)
      - nenhum split válido restante

    Todo nó (interno ou folha) guarda `prediction` = classe majoritária
    e `n_samples`, necessários para pós-poda (REP / CCP).
    """
    y = df[target]
    maj = majority_class(y)
    n = len(df)

    pure = y.nunique() == 1
    too_small = n < min_samples_split
    too_deep = max_depth is not None and depth >= max_depth

    if pure or too_small or too_deep:
        return Node(prediction=maj, n_samples=n)

    feature, threshold, _ = find_best_split(df, target)
    if feature is None:
        return Node(prediction=maj, n_samples=n)

    left_df = df[df[feature] <= threshold]
    right_df = df[df[feature] > threshold]

    # Split degenerado (não deve ocorrer se find_cart_threshold estiver ok)
    if len(left_df) == 0 or len(right_df) == 0:
        return Node(prediction=maj, n_samples=n)

    left = build_tree(
        left_df,
        target=target,
        depth=depth + 1,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
    )
    right = build_tree(
        right_df,
        target=target,
        depth=depth + 1,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
    )

    return Node(
        feature=feature,
        threshold=threshold,
        left=left,
        right=right,
        prediction=maj,
        n_samples=n,
    )


def print_tree(node, indent=0):
    """Impressão legível da árvore para inspeção."""
    pad = "  " * indent
    if node.is_leaf:
        print(f"{pad}→ predict {node.prediction} (n={node.n_samples})")
        return

    print(f"{pad}if {node.feature} <= {node.threshold}:  # n={node.n_samples}")
    print_tree(node.left, indent + 1)
    print(f"{pad}else:  # {node.feature} > {node.threshold}")
    print_tree(node.right, indent + 1)


# Exemplo: árvore rasa para visualizar a estrutura
tree = build_tree(df_train, max_depth=3)
print_tree(tree)


# 06. Estudo de Conceitos

## Algoritmos de Árvores de Decisão

Esta implementação segue a lógica do **CART** (*Classification and Regression Trees*, Breiman et al.).

| Algoritmo | Ideia central | Split | Critério típico | Observação |
|-----------|---------------|-------|-----------------|------------|
| **ID3** | Indução gulosa com ganho de informação | Multiway em categóricas | Entropia / Information Gain | Não trata bem contínuas “de fábrica”; tendência a favorecer atributos com muitos níveis |
| **C4.5** | Sucessor do ID3 | Contínuas + categóricas; lida com missing | **Gain Ratio** (corrige viés do IG puro) | Muito usado pedagogicamente; poda (pessimistic pruning) |
| **CART** | Particionamento binário recursivo | **Sempre binário** (`≤` vs `>`) | Classificação: **Gini** (ou entropia); Regressão: **MSE**/variância | Mesma estrutura para classificar e regressão; base de Random Forest etc. |

Pontos práticos:
- “Árvore de decisão” não é um único algoritmo: mudam **critério de impureza**, **forma do split** e **estratégia de poda**.
- Nosso código: splits binários + Gini → alinhado ao CART de classificação.

## Lidando com Variáveis Nominais

No dataset deste exercício, `Sex` é nominal **binária** (0/1). Com só dois valores, um threshold `≤` não inventa uma ordem problemática: o único corte útil separa as duas classes da feature.

Já com variáveis nominais de **3 ou mais** níveis *sem* ordem natural (ex.: porto de embarque), o `≤` do CART **impõe uma ordem artificial**. Isso pode:
- gerar splits semanticamente estranhos, ou
- exigir vários níveis da árvore para aprender partições que um split categórico nativo faria de uma vez.

Abordagens comuns:
1. **One-hot encoding** — cada categoria vira pergunta sim/não; compatível com CART binário (o que usamos).
2. **Tratar como ordinal** — só se a ordem for real (ex.: `Pclass` 1 ≺ 2 ≺ 3).
3. **Split categórico nativo** (CART/algumas libs) — testa partições de conjuntos de níveis.

## Bias e Variance

Árvores ilustram bem o tradeoff bias–variance:

- **Árvores rasas** (pouca profundidade): modelo simples → em geral **alto bias**, **baixa variance** (subajustam; erram de forma parecida em vários conjuntos de treino).
- **Árvores profundas** (até pureza / depth “infinita” no treino): encaixam o treino quase perfeitamente → **baixo bias no treino**, **alta variance** (mudam muito com amostras diferentes; generalizam mal).

Por isso `max_depth`, `min_samples_leaf` e, depois, **pré-poda / pós-poda** existem: não para “deixar a árvore mais certa no treino”, e sim para **controlar complexidade** e melhorar desempenho em dados não vistos.

Regra mental: *profundidade ↑ ⇒ flexibilidade ↑ ⇒ risco de overfitting ↑*.

# 07. Pós-poda: REP e CCP

Após crescer a árvore, a **pós-poda** remove subárvores que não melhoram a generalização.

- **REP (Reduced-Error Pruning):** usa um conjunto de *validação*; de baixo para cima, vira folha se o erro na validação não piora.
- **CCP (Cost-Complexity Pruning):** penaliza o número de folhas: $R_\alpha(T) = R(T) + \alpha |\tilde{T}|$. Gera uma sequência de árvores e escolhe $\alpha$ (ou a árvore) pela validação.


In [ ]:
import copy


# ---------------------------------------------------------------------------
# Predição e métricas
# ---------------------------------------------------------------------------

def predict_row(node, row):
    """Percorre a árvore até uma folha para uma única observação (Series/dict)."""
    while not node.is_leaf:
        if row[node.feature] <= node.threshold:
            node = node.left
        else:
            node = node.right
    return node.prediction


def predict(node, df):
    """Predições para um DataFrame."""
    return df.apply(lambda row: predict_row(node, row), axis=1)


def accuracy(node, df, target="Survived"):
    if len(df) == 0:
        return 0.0
    y_hat = predict(node, df)
    return float((y_hat.values == df[target].values).mean())


def count_leaves(node):
    if node.is_leaf:
        return 1
    return count_leaves(node.left) + count_leaves(node.right)


def clone_tree(node):
    """Cópia profunda — CCP/REP não devem destruir a árvore original."""
    return copy.deepcopy(node)


# ---------------------------------------------------------------------------
# REP — Reduced-Error Pruning
# ---------------------------------------------------------------------------

def _validation_errors(node, df_val, target):
    """Erros de classificação da subárvore vs. da folha majoritária, no DF de validação."""
    if len(df_val) == 0:
        return 0, 0

    y = df_val[target]
    leaf_pred = node.prediction
    errors_as_leaf = int((y != leaf_pred).sum())

    if node.is_leaf:
        return errors_as_leaf, errors_as_leaf

    y_hat = predict(node, df_val)
    errors_subtree = int((y_hat.values != y.values).sum())
    return errors_subtree, errors_as_leaf


def reduced_error_prune(node, df_val, target="Survived"):
    """
    Pós-poda bottom-up: poda o nó se a folha não aumentar o erro na validação.
    Modifica a árvore in-place; use clone_tree antes se quiser preservar a original.
    """
    if node.is_leaf:
        return node

    # Garante que a validação siga o mesmo split do nó
    left_val = df_val[df_val[node.feature] <= node.threshold]
    right_val = df_val[df_val[node.feature] > node.threshold]

    reduced_error_prune(node.left, left_val, target)
    reduced_error_prune(node.right, right_val, target)

    errors_subtree, errors_as_leaf = _validation_errors(node, df_val, target)
    if errors_as_leaf <= errors_subtree:
        node.prune_to_leaf()

    return node


# ---------------------------------------------------------------------------
# CCP — Cost-Complexity Pruning (weakest-link)
# ---------------------------------------------------------------------------

def _node_misclassification(node, df, target):
    """Nº de erros se este nó fosse folha, nas amostras de `df` que caem nele."""
    if len(df) == 0:
        return 0
    return int((df[target] != node.prediction).sum())


def _subtree_misclassification(node, df, target):
    """Nº de erros da subárvore completa em `df`."""
    if len(df) == 0:
        return 0
    if node.is_leaf:
        return _node_misclassification(node, df, target)
    left_df = df[df[node.feature] <= node.threshold]
    right_df = df[df[node.feature] > node.threshold]
    return (
        _subtree_misclassification(node.left, left_df, target)
        + _subtree_misclassification(node.right, right_df, target)
    )


def _compute_weakest_link(node, df, target, n_train):
    """
    Para cada nó interno, g(t) = (R(t) - R(T_t)) / (|T_t| - 1),
    com R = taxa de erro no treino (erros / n_train).
    Retorna (menor_g, nó_correspondente).
    """
    if node.is_leaf:
        return float("inf"), None

    left_df = df[df[node.feature] <= node.threshold]
    right_df = df[df[node.feature] > node.threshold]

    g_left, node_left = _compute_weakest_link(node.left, left_df, target, n_train)
    g_right, node_right = _compute_weakest_link(node.right, right_df, target, n_train)

    r_node = _node_misclassification(node, df, target) / n_train
    r_subtree = _subtree_misclassification(node, df, target) / n_train
    n_leaves = count_leaves(node)

    if n_leaves <= 1:
        g_here = float("inf")
    else:
        g_here = (r_node - r_subtree) / (n_leaves - 1)

    # Empate: prefere o menor g; se empatar, qualquer um (aqui: este nó)
    best_g, best_node = g_here, node
    if g_left < best_g:
        best_g, best_node = g_left, node_left
    if g_right < best_g:
        best_g, best_node = g_right, node_right

    return best_g, best_node


def cost_complexity_prune_path(tree, df_train, target="Survived"):
    """
    Gera a sequência de poda CCP: lista de (alpha_efetivo, árvore_clonada).

    alpha_efetivo ≈ g(t) do weakest link podado para obter aquela árvore.
    A primeira entrada é a árvore original com alpha=0.
    """
    n_train = len(df_train)
    current = clone_tree(tree)
    path = [(0.0, clone_tree(current))]

    while not current.is_leaf:
        g, weakest = _compute_weakest_link(current, df_train, target, n_train)
        if weakest is None or g == float("inf"):
            break
        weakest.prune_to_leaf()
        path.append((float(g), clone_tree(current)))

    return path


def cost_complexity_prune(tree, df_train, df_val, target="Survived"):
    """
    CCP completo: constrói o path e escolhe a árvore com melhor accuracy na validação.
    Em empate, prefere a árvore com menos folhas (mais regularizada).
    """
    path = cost_complexity_prune_path(tree, df_train, target)

    best = None
    best_acc = -1.0
    best_leaves = float("inf")
    best_alpha = 0.0

    rows = []
    for alpha, pruned in path:
        acc = accuracy(pruned, df_val, target)
        leaves = count_leaves(pruned)
        rows.append({"alpha": alpha, "leaves": leaves, "val_accuracy": acc})
        if (acc > best_acc) or (acc == best_acc and leaves < best_leaves):
            best_acc = acc
            best_leaves = leaves
            best = pruned
            best_alpha = alpha

    summary = pd.DataFrame(rows)
    return best, best_alpha, summary


In [ ]:
# --- Demonstração: árvore cheia + REP + CCP ---

# Hold-out interno do treino (validação só para escolher a poda; não misturar com teste final)
df_fit = df_train.sample(frac=0.75, random_state=42)
df_val = df_train.drop(df_fit.index)

# Árvore "cheia" (só para no mínimo 2 amostras para split — sem max_depth)
tree_full = build_tree(df_fit, min_samples_split=2)

tree_rep = clone_tree(tree_full)
reduced_error_prune(tree_rep, df_val)

tree_ccp, alpha_star, ccp_summary = cost_complexity_prune(tree_full, df_fit, df_val)

print("Folhas | acc_fit | acc_val")
for name, t in [("full", tree_full), ("REP", tree_rep), ("CCP", tree_ccp)]:
    print(
        f"{name:4s}: {count_leaves(t):3d} | "
        f"{accuracy(t, df_fit):.3f}   | {accuracy(t, df_val):.3f}"
    )

print(f"\nCCP: alpha* ≈ {alpha_star:.6f}")
print("\nPath CCP (accuracy na validação):")
display(ccp_summary)

print("\n--- Árvore após REP ---")
print_tree(tree_rep)
print("\n--- Árvore após CCP ---")
print_tree(tree_ccp)
